# Coarse Fog Masking 實作與測試 (真實影片版)

這份 Notebook 實作了專案 Proposal 中定義的 **Coarse Fog Masking (粗略遮罩)**。利用局部對比度、HSV 飽和度以及影像梯度，透過聯集投票機制來找出初步的起霧像素。

**成員 A 的任務**：請將真實的起霧影片放入資料夾中，修改下方 Cell 的 `video_path`，並不斷微調 `thresh_var`, `thresh_sat`, `thresh_grad` 這三個參數，直到遮罩能完美捕捉到水氣且不誤判正常場景為止。

In [10]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import warnings

warnings.simplefilter("ignore", category=RuntimeWarning)

# 設定 matplotlib 顯示大小
plt.rcParams['figure.figsize'] = [18, 12]

part 1: coarse fog mask

### 核心函式定義

In [11]:

def get_coarse_fog_mask(image_bgr, patch_size=15, w_dcp=0.5, w_grad=0.3, w_sat=0.2):
    """
    保留您驗證過完美的機率型物理遮罩邏輯
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV).astype(np.float32) / 255.0
    
    min_channel = np.min(image_bgr.astype(np.float32) / 255.0, axis=2)
    kernel_dcp = cv2.getStructuringElement(cv2.MORPH_RECT, (patch_size, patch_size))
    conf_dcp = cv2.erode(min_channel, kernel_dcp)
    
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobelx**2 + sobely**2)
    grad_norm = np.clip(grad_mag / (np.percentile(grad_mag, 95) + 1e-6), 0, 1)
    conf_grad = 1.0 - grad_norm
    
    sat = hsv[:, :, 1]
    conf_sat = 1.0 - sat
    
    total_w = w_dcp + w_grad + w_sat
    confidence_map = ((w_dcp/total_w) * conf_dcp + 
                      (w_grad/total_w) * conf_grad + 
                      (w_sat/total_w) * conf_sat)
    
    val = hsv[:, :, 2]
    confidence_map = confidence_map * np.clip(val * 2.0, 0, 1) 
    confidence_map = cv2.GaussianBlur(confidence_map, (101, 101), 0)
    
    binary_mask = (confidence_map > 0.6).astype(np.uint8) * 255
    morph_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, morph_kernel)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, morph_kernel)
    
    return binary_mask, confidence_map


def compute_affine_matrix(kp_curr, kp_ref, good_matches):
    if len(good_matches) < 15: return None
    src_pts = np.float32([kp_curr[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp_ref[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    affine_matrix, _ = cv2.estimateAffinePartial2D(src_pts, dst_pts, method=cv2.RANSAC, ransacReprojThreshold=3.0)
    
    if affine_matrix is not None:
        # 🛡️ 彈性約束：防止瘋狂縮放，但允許鏡頭微小呼吸 (0.98 ~ 1.02)
        scale = np.sqrt(affine_matrix[0, 0]**2 + affine_matrix[1, 0]**2)
        clamped_scale = np.clip(scale, 0.98, 1.02)
        affine_matrix[0:2, 0:2] = (affine_matrix[0:2, 0:2] / scale) * clamped_scale
        
        H_temp = np.eye(3, dtype=np.float32)
        H_temp[0:2, :] = affine_matrix
        return H_temp
    return None


def smooth_trajectory(H_list, window_size=9):
    """
    軌跡平滑濾波器：使用 Moving Average 消除 RANSAC 帶來的亞像素高頻震動。
    """
    if len(H_list) < window_size:
        return H_list
        
    # 提取參數: tx, ty, theta, scale
    params = np.array([[H[0, 2], H[1, 2], np.arctan2(H[1, 0], H[0, 0]), np.sqrt(H[0, 0]**2 + H[1, 0]**2)] for H in H_list])
    smoothed_params = np.copy(params)
    
    kernel = np.ones(window_size) / window_size
    pad_size = window_size // 2
    
    for i in range(4):
        # 使用鏡像填充避免頭尾資料突變
        padded = np.pad(params[:, i], (pad_size, pad_size), mode='reflect')
        smoothed_params[:, i] = np.convolve(padded, kernel, mode='valid')
        
    smoothed_H_list = []
    for i in range(len(H_list)):
        tx, ty, theta, scale = smoothed_params[i]
        H = np.eye(3, dtype=np.float32)
        H[0, 0] = scale * np.cos(theta)
        H[0, 1] = -scale * np.sin(theta)
        H[1, 0] = scale * np.sin(theta)
        H[1, 1] = scale * np.cos(theta)
        H[0, 2] = tx
        H[1, 2] = ty
        smoothed_H_list.append(H)
        
    return smoothed_H_list


def align_video_keyframe_anchored(video_path, total_frames):
    """
    關鍵幀錨定對齊：大幅減少矩陣連乘的累積誤差 (解決旋轉扭曲)。
    """
    cap = cv2.VideoCapture(video_path)
    ret, frame_curr = cap.read()
    h, w = frame_curr.shape[:2]
    
    sift = cv2.SIFT_create()
    bf = cv2.BFMatcher()
    
    def process_frame(frame):
        mask, _ = get_coarse_fog_mask(frame)
        valid_mask = cv2.bitwise_not(mask)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return sift.detectAndCompute(gray, mask=valid_mask)
        
    anchor_kp, anchor_des = process_frame(frame_curr)
    H_anchor_to_global = np.eye(3, dtype=np.float32)
    
    prev_kp, prev_des = anchor_kp, anchor_des
    raw_H_list = [np.eye(3, dtype=np.float32)]
    
    print("🔗 執行關鍵幀錨定對齊 (降低累積扭曲)...")
    for i in tqdm(range(1, total_frames)):
        ret, frame_curr = cap.read()
        if not ret: break
        
        curr_kp, curr_des = process_frame(frame_curr)
        matches = bf.knnMatch(curr_des, anchor_des, k=2)
        good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]
        
        H_curr_to_anchor = compute_affine_matrix(curr_kp, anchor_kp, good_matches)
        
        # 判斷是否需要更換錨點 (位移過大或特徵丟失)
        drift_x = abs(H_curr_to_anchor[0, 2]) if H_curr_to_anchor is not None else 0
        drift_y = abs(H_curr_to_anchor[1, 2]) if H_curr_to_anchor is not None else 0
        
        if H_curr_to_anchor is None or len(good_matches) < 30 or drift_x > w * 0.15 or drift_y > h * 0.15:
            # ⚓ 更換錨點到上一幀！
            anchor_kp, anchor_des = prev_kp, prev_des
            H_anchor_to_global = raw_H_list[-1]
            
            # 重新比對當前幀與新錨點
            matches = bf.knnMatch(curr_des, anchor_des, k=2)
            good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]
            H_curr_to_anchor = compute_affine_matrix(curr_kp, anchor_kp, good_matches)
            
        if H_curr_to_anchor is not None:
            H_curr_to_global = H_anchor_to_global @ H_curr_to_anchor
        else:
            H_curr_to_global = raw_H_list[-1] # 對齊徹底失敗則假裝沒動
            
        raw_H_list.append(H_curr_to_global)
        prev_kp, prev_des = curr_kp, curr_des
        
    cap.release()
    
    # 🌟 核心修復：對生硬且帶有 RANSAC 噪聲的軌跡進行強效平滑濾波，消除修補區域的抖動！
    print("🌊 執行軌跡平滑濾波 (消除高頻抖動)...")
    raw_H_list = smooth_trajectory(raw_H_list, window_size=9)
    
    # 精算全域大畫布尺寸
    print("📐 計算全域大畫布尺寸...")
    corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
    all_corners = np.concatenate([cv2.perspectiveTransform(corners, H) for H in raw_H_list], axis=0)
    
    [x_min, y_min] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
    [x_max, y_max] = np.int32(all_corners.max(axis=0).ravel() + 0.5)
    pano_w, pano_h = x_max - x_min, y_max - y_min
    
    T_offset = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]], dtype=np.float32)
    final_H_list = [T_offset @ H for H in raw_H_list]
    
    print(f"✅ 畫布尺寸: {pano_w}x{pano_h}")
    return final_H_list, pano_w, pano_h


def process_global_median_fusion(video_path, H_list, pano_w, pano_h, total_frames):
    """
    使用 NaN Median (中值濾波) 來建造畫布。
    捨棄會導致模糊的平均值，保證遠景細節像刀切一樣銳利！
    """
    cap = cv2.VideoCapture(video_path)
    
    # 決定抽樣率：最多抓取 30 幀作為樣本來算中值，防止記憶體爆掉
    sample_step = max(1, total_frames // 120)
    sample_indices = list(range(0, total_frames, sample_step))
    num_samples = len(sample_indices)
    
    print(f"\n[Phase 1] 均勻抽取 {num_samples} 個關鍵幀構建極銳利中值畫布...")
    
    # 建立巨大浮點數矩陣 (N, H, W, 3) 準備儲存樣本
    # 填入 np.nan (Not a Number)，這樣有霧的地方就不會參與中值投票
    canvas_samples = np.full((num_samples, pano_h, pano_w, 3), np.nan, dtype=np.float32)
    
    sample_idx = 0
    for t in tqdm(range(total_frames), desc="投射清晰像素"):
        ret, frame = cap.read()
        if not ret: break
            
        if t not in sample_indices:
            continue
            
        fog_mask, _ = get_coarse_fog_mask(frame)
        clean_mask = (255 - fog_mask) > 128 # True 代表乾淨
        
        H = H_list[t]
        warped_frame = cv2.warpPerspective(frame, H, (pano_w, pano_h), flags=cv2.INTER_LANCZOS4).astype(np.float32)
        warped_clean_mask = cv2.warpPerspective(clean_mask.astype(np.uint8), H, (pano_w, pano_h))
        
        # 將邊緣侵蝕避免黑邊污染
        warped_clean_mask = cv2.erode(warped_clean_mask, np.ones((5, 5), np.uint8), iterations=3)
        valid_mask_2d = warped_clean_mask > 0
        
        # 只把「有效且無霧」的像素填入這層畫布
        canvas_samples[sample_idx][valid_mask_2d] = warped_frame[valid_mask_2d]
        sample_idx += 1

    print("\n🧠 正在執行核心：強健截尾百分位融合 (Robust Trimmed Percentile)...")
    with warnings.catch_warnings():
        global_median_bg = np.nanmedian(canvas_samples, axis=0)
        
    # 如果有些地方從頭到尾都是霧(全 NaN)，補 0 防錯
    global_median_bg = np.nan_to_num(global_median_bg, nan=0.0).astype(np.uint8)
    
	# === 插入在這裡 (Phase 1 結尾) ===
    # 執行 Unsharp Mask (反銳化掩蔽) 來補償視差造成的紋理流失
    # 公式：Sharp = Original + (Original - Blurred) * amount
    blurred_bg = cv2.GaussianBlur(global_median_bg, (0, 0), 2.0)
    global_median_bg = cv2.addWeighted(global_median_bg, 1.5, blurred_bg, -0.5, 0)
    
    # 輸出畫布供檢查，這張圖應該會極度銳利！
    cv2.imwrite('ultra_sharp_median_canvas.jpg', global_median_bg)
    print("✅ 完美畫布已儲存為 ultra_sharp_median_canvas.jpg")

    # ==========================================
    # Phase 2: 最終渲染拉回 (Repainting)
    # ==========================================
    print("\n[Phase 2] 從銳利畫布拉回並修補影片...")
    final_dehazed_frames = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    
    for t in tqdm(range(total_frames), desc="渲染中"):
        ret, frame_raw = cap.read()
        if not ret: break
            
        raw_h, raw_w = frame_raw.shape[:2]
        ret_inv, H_inv = cv2.invert(H_list[t])
        if not ret_inv: H_inv = np.eye(3)
            
        bg_at_current_view = cv2.warpPerspective(global_median_bg, H_inv, (raw_w, raw_h), flags=cv2.INTER_LANCZOS4)
        
        mask_coarse, _ = get_coarse_fog_mask(frame_raw) 
        mask_float = cv2.GaussianBlur(mask_coarse.astype(np.float32)/255.0, (81, 81), 0)
        
        # 🌟 核心修復：拔除導致邊緣鬼影殘留與抖動的 prev_mask EMA 時域平滑！
        # 直接使用巨大高斯羽化的當下完美遮罩，避免錯位干擾。
        mask_3c = np.stack([mask_float]*3, axis=-1)
        
        valid_bg_2d = np.any(bg_at_current_view > 0, axis=-1).astype(np.uint8)
        valid_bg_2d = cv2.erode(valid_bg_2d, np.ones((5, 5), np.uint8), iterations=3)
        valid_bg_float = cv2.GaussianBlur(valid_bg_2d.astype(np.float32), (41, 41), 0)
        valid_bg_mask = np.expand_dims(valid_bg_float, axis=-1)
        
        final_mask = mask_3c * valid_bg_mask
        
        blended = frame_raw * (1.0 - final_mask) + bg_at_current_view * final_mask
        blended = np.clip(blended, 0, 255).astype(np.uint8)
        
        final_dehazed_frames.append(blended)
        
    cap.release()
    return final_dehazed_frames

### 讀取真實影片與視覺化
請將 `video_path` 替換為你們手邊的影片路徑。你可以透過改變 `frame_id` 來隨機抽查影片中的不同時間點，確保參數在整支影片中都具備泛化能力。

part 2: homology transformation

part 3: temporal fusion & repainting

In [12]:

if __name__ == "__main__":
    video_path = './data/foggy_vid/1.mp4'  
    
    if not os.path.exists(video_path):
        print(f"找不到影片檔案：{video_path}")
    else:
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        print(f"🎬 影片總幀數為: {total_frames}")

        # 1. 執行關鍵幀錨定對齊 (無區塊斷層、少累積扭曲)
        H_list, p_w, p_h = align_video_keyframe_anchored(video_path, total_frames)

        if H_list:
            # 2. 執行全域中值建圖與融合 (解決平均法模糊)
            dehazed_video = process_global_median_fusion(video_path, H_list, p_w, p_h, total_frames)

            # 3. 儲存結果
            frame_h, frame_w = dehazed_video[0].shape[:2]
            out_path = 'dehazed_output_global_median.mp4'
            out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), 30, (frame_w, frame_h))
            for f in dehazed_video: out.write(f)
            out.release()
            
            print(f"\n🎉 大功告成！極度銳利的去霧影片已儲存至：{out_path}")

🎬 影片總幀數為: 347
🔗 執行關鍵幀錨定對齊 (降低累積扭曲)...


100%|██████████| 346/346 [00:37<00:00,  9.13it/s]


🌊 執行軌跡平滑濾波 (消除高頻抖動)...
📐 計算全域大畫布尺寸...
✅ 畫布尺寸: 1602x855

[Phase 1] 均勻抽取 174 個關鍵幀構建極銳利中值畫布...


投射清晰像素: 100%|██████████| 347/347 [00:13<00:00, 25.66it/s]



🧠 正在執行核心：強健截尾百分位融合 (Robust Trimmed Percentile)...
✅ 完美畫布已儲存為 ultra_sharp_median_canvas.jpg

[Phase 2] 從銳利畫布拉回並修補影片...


渲染中: 100%|██████████| 347/347 [00:28<00:00, 12.00it/s]



🎉 大功告成！極度銳利的去霧影片已儲存至：dehazed_output_global_median.mp4
